In [188]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from numpy.ma.extras import average
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.utils import make_grid

import numpy as np
from time import time
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
from sklearn.metrics import confusion_matrix
import sys

# Lecture 14: Import MNIST Images

In [189]:
transform = transforms.ToTensor()
train_data = datasets.MNIST(root=r"./Datasets/MNIST", train=True, download=True, transform=transform)
test_data = datasets.MNIST(root=r"./Datasets/MNIST", train=False, download=True, transform=transform)
train_data, test_data

(Dataset MNIST
     Number of datapoints: 60000
     Root location: ./Datasets/MNIST
     Split: Train
     StandardTransform
 Transform: ToTensor(),
 Dataset MNIST
     Number of datapoints: 10000
     Root location: ./Datasets/MNIST
     Split: Test
     StandardTransform
 Transform: ToTensor())

# Lecture 15: Convolutional and Pooling Layers

## Quick Note for Convolutional and Pooling Layers:
1. Out put size equation:<center>$O=\frac{W-K+2{\times}P}{S}+1$</center>
    * O: Output Size
    * W: input Volume
    * K: Kernel Size
    * P: Padding
    * S: Stride
2. If padding is not set, we will lose image boundary pixels.
3. Max pooling will round down if output size has remainder.

## Section 1: Create a Batch Size for Images
Batch size refers to the number of individual data samples that are processed together in a single forward pass and backward pass during the training of a neural network.

In [190]:
batch_size = 512
train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_data, batch_size=batch_size, shuffle=False)

## Section 2: Define CNN Model by Describing Convolutional Layers

In [191]:
conv1 = nn.Conv2d(in_channels=1, out_channels=6, kernel_size=3, stride=1)
conv2 = nn.Conv2d(in_channels=6, out_channels=16, kernel_size=3, stride=1)

## Section 3: Grab 1 MNIST record/image.

In [192]:
for idx, (X_train, y_train) in enumerate(train_data):
    break

X = X_train.reshape(1, 1, 28, 28)
X.shape

torch.Size([1, 1, 28, 28])

## Section 4: Pass Through 2 Convolutional Layers

In [193]:
X = F.relu(conv1(X))
print(X.shape)

X = F.relu(conv2(X))
print(X.shape)

torch.Size([1, 6, 26, 26])
torch.Size([1, 16, 24, 24])


## Section 5: Pass Through Pooling Layer

In [194]:
X = F.max_pool2d(X, kernel_size=2, stride=2)
X.shape

torch.Size([1, 16, 12, 12])

# Lecture 16: Convolutional Neural Network Model

## Section 1: Config CNN Model
* Create Convolutional Layers and Pooling Layers
* Connect all Layers

In [195]:
class ConvolutionalNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=6, kernel_size=3, stride=1)
        self.conv2 = nn.LazyConv2d(out_channels=16, kernel_size=3, stride=1)

        self.fc1 = nn.LazyLinear(out_features=120)
        self.fc2 = nn.LazyLinear(out_features=80)
        self.fc3 = nn.LazyLinear(out_features=10)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = F.max_pool2d(x, kernel_size=2, stride=2)
        x = x.flatten(start_dim=1)

        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = F.log_softmax(x, dim=1)

        return x

## Section 2: Create an Instance of CNN Model

In [196]:
torch.manual_seed(42)
# device = torch.device("mps") if torch.backends.mps.is_available() else torch.device("cpu")
device = torch.device("cpu")
model = ConvolutionalNetwork().to(device)
device

device(type='cpu')

## Section 3: Create Loss Function and Optimizor

In [197]:
criterion = nn.CrossEntropyLoss()
optimizor = torch.optim.Adam(model.parameters(), lr=0.001)

# Lecture 17: Train and Test CNN Network

## Section 1: Create Tracking Variables

In [198]:
epochs = 30
total_train_losses = []
total_test_losses = []

## Section 2: Train CNN Model

In [199]:
for idx in range(epochs):
    train_losses = []

    t0 = time()
    pbar_train = tqdm(train_loader, desc=f"Epoch {idx+1}",unit="batch", file=sys.stdout)
    for batch, (X_train, y_train) in enumerate(pbar_train):
        X_train = X_train.to(device)
        y_train = y_train.to(device)

        y_pred = model.forward(X_train)
        loss = criterion(y_pred, y_train)
        train_losses.append(loss.item())

        optimizor.zero_grad()
        loss.backward()
        optimizor.step()

        run_time = time() - t0
        pbar_train.set_postfix({"loss": f"{np.mean(train_losses):.4f}", "Training Time": f"{run_time:.3f}s"})

    total_train_losses.append(np.mean(train_losses))

Epoch 30: 100%|██████████| 118/118 [00:19<00:00,  5.92batch/s, loss=0.4368, Training Time=19.921s]


## Section 3: Test CNN Model

In [200]:
pbar_test = tqdm(test_loader, desc="Test Progress", unit="batch", file=sys.stdout)

with torch.no_grad():
    test_losses = []
    t0 = time()
    for batch, (X_test, y_test) in enumerate(pbar_test):
        X_test, y_test = X_test.to(device), y_test.to(device)
        y_eval = model(X_test)
        loss = criterion(y_eval, y_test)
        test_losses.append(loss.item())
        run_time = time() - t0
        pbar_test.set_postfix({"Loss": f"{np.mean(test_losses):.4f}", "Test Time": f"{run_time:.3f}s"})

Test Progress: 100%|██████████| 20/20 [00:02<00:00,  8.40batch/s, Loss=0.4749, Test Time=2.380s]


# Lecture 18: Graph CNN Model Result

In [201]:
torch.save(model.state_dict(), r"./MNIST.pt")